# Entity Reviews Analysis

This notebook analyzes review sentiment and entity relevance for the entity-driven app.
- Review-level sentiment (multilingual).
- Entity-match score for selected entities and suggested new entities per review.


In [1]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

DB_PATH = Path('../data/app.sqlite')
if not DB_PATH.exists():
    raise FileNotFoundError(f'DB not found at {DB_PATH.resolve()}')

conn = sqlite3.connect(DB_PATH)
reviews = pd.read_sql('SELECT id, content, created_at, updated_at, user_id FROM review', conn)
nodes = pd.read_sql('SELECT id, name, type FROM nodes', conn)
review_entities = pd.read_sql('SELECT review_id, node_id, alias FROM review_entity', conn)
conn.close()

print('reviews', reviews.shape)
print('nodes', nodes.shape)
print('review_entities', review_entities.shape)


reviews (25, 5)
nodes (47, 3)
review_entities (34, 3)


In [7]:
#reviews
#nodes
review_entities

,review_id,node_id,alias
0,1,1,BHC 쏘마치
1,2,2,처가집 양념통닭 K마라치킨
2,5,6,노랑통닭
3,3,3,멕시카나
4,3,9,배달의민족
5,4,10,비비큐 맵소디
6,6,11,푸라닭 깐풍치킨
7,7,12,교촌치킨 간장/레드 반반
8,8,13,자담치킨
9,8,14,땡겨요


## 1) Review-level sentiment (multilingual transformer)

We use a multilingual RoBERTa sentiment model and run it over full reviews (not sentence-by-sentence).


In [2]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

sentiment_model_id = 'cardiffnlp/twitter-xlm-roberta-base-sentiment'
tokenizer = AutoTokenizer.from_pretrained(sentiment_model_id)
model = AutoModelForSequenceClassification.from_pretrained(
    sentiment_model_id,
    use_safetensors=True
)
sentiment_pipe = pipeline(
    'sentiment-analysis',
    model=model,
    tokenizer=tokenizer,
    truncation=True,
    max_length=256,
    batch_size=16
)

def analyze_sentiments(texts):
    # Keep order aligned with input
    results = sentiment_pipe(texts)
    return pd.DataFrame(results)

sentiment_df = analyze_sentiments(reviews['content'].tolist())
sentiment_df = pd.concat([reviews[['id', 'content']], sentiment_df], axis=1)
sentiment_df.rename(columns={'id': 'review_id'}, inplace=True)
sentiment_df.head()


Device set to use cpu


,review_id,content,label,score
0,1,BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마...,positive,0.530003
1,2,처가집 양념통닭에서 마라치킨 주문. 그냥 달달한 양념 치킨에 마라향 살짝 추가한 정...,neutral,0.379968
2,3,배달앱에서 멕시카나 치킨 할인한다기에 주문하려했더니 미금점 영업을 하지 않는다. 이...,negative,0.635604
3,4,비비큐에서 맵소디 배달 주문. 양념 맛은 기억나지 않지만 소스 안쪽에서도 바삭한 식...,neutral,0.572736
4,5,노랑통닭은 순살도 가격이 더 비싸지 않아 좋다. 튀김 부스러기는 과자같이 바삭하니 ...,positive,0.694385


In [3]:
sentiment_df['label'].value_counts(normalize=True).rename('share')


label
negative    0.48
positive    0.32
neutral     0.20
Name: share, dtype: float64

## 2) Entity match + suggestions

We embed review text and entity labels, then compute cosine similarity.
- `selected_match_score`: mean similarity between the review and its selected entities.
- `suggested_entities`: top-k entities not already selected.


In [4]:
embedder_id = 'paraphrase-multilingual-MiniLM-L12-v2'
embedder = SentenceTransformer(embedder_id)

nodes = nodes.copy()
nodes['label'] = nodes.apply(lambda row: f"{row['name']} ({row['type']})", axis=1)

entity_labels = nodes['label'].tolist()
entity_embeddings = embedder.encode(entity_labels, convert_to_tensor=True, normalize_embeddings=True)

review_texts = reviews['content'].tolist()
review_embeddings = embedder.encode(review_texts, convert_to_tensor=True, normalize_embeddings=True)

review_entities_grouped = (
    review_entities.groupby('review_id')['node_id']
    .apply(list)
)

node_id_to_index = {node_id: i for i, node_id in enumerate(nodes['id'].tolist())}

def score_selected_entities(review_id, review_idx):
    node_ids = review_entities_grouped.get(review_id, [])
    if not node_ids:
        return np.nan
    indices = [node_id_to_index[nid] for nid in node_ids if nid in node_id_to_index]
    if not indices:
        return np.nan
    sims = util.cos_sim(review_embeddings[review_idx], entity_embeddings[indices]).cpu().numpy()
    return float(sims.mean())

def suggest_entities(review_id, review_idx, k=5):
    selected = set(review_entities_grouped.get(review_id, []))
    sims = util.cos_sim(review_embeddings[review_idx], entity_embeddings).cpu().numpy().ravel()
    ranked = sims.argsort()[::-1]
    suggestions = []
    for idx in ranked:
        node_id = nodes.iloc[idx]['id']
        if node_id in selected:
            continue
        suggestions.append({
            'node_id': int(node_id),
            'label': nodes.iloc[idx]['label'],
            'score': float(sims[idx])
        })
        if len(suggestions) >= k:
            break
    return suggestions

rows = []
for idx, row in reviews.iterrows():
    review_id = row['id']
    rows.append({
        'review_id': review_id,
        'selected_match_score': score_selected_entities(review_id, idx),
        'suggested_entities': suggest_entities(review_id, idx, k=5)
    })

suggestions_df = pd.DataFrame(rows)
suggestions_df.head()


,review_id,selected_match_score,suggested_entities
0,1,0.479157,"[{'node_id': 54, 'label': '닭강정 (menu)', 'score..."
1,2,0.252874,"[{'node_id': 54, 'label': '닭강정 (menu)', 'score..."
2,3,0.268003,"[{'node_id': 34, 'label': '정동마트 (vendor)', 'sc..."
3,4,0.388947,"[{'node_id': 47, 'label': '분당맛소 (vendor)', 'sc..."
4,5,0.638946,"[{'node_id': 54, 'label': '닭강정 (menu)', 'score..."


In [ ]:
df = suggestions_df.sort_values('selected_match_score')

In [15]:
review_id = 16
df.loc[review_id, 'suggested_entities']

[{'node_id': 29, 'label': '한사랑 정형외과의원 (vendor)', 'score': 0.46180397272109985},
 {'node_id': 30, 'label': '이주철마취과의원 (vendor)', 'score': 0.2528977394104004},
 {'node_id': 13, 'label': '자담치킨 (vendor)', 'score': 0.09404535591602325},
 {'node_id': 20, 'label': '굽네치킨 (vendor)', 'score': 0.07251551747322083},
 {'node_id': 43, 'label': '교촌치킨 (vendor)', 'score': 0.07195234298706055}]

In [16]:
reviews.set_index('id').loc[review_id, 'content']

'발뒤꿈치가 아파 정형외과를 찾는데 저번에 비싼 주사 맞은데를 피해서 다른 병원을 갔다. 갔더니 아픈 부위를 직접 보지도 않고 발모형에서 아픈 부위를 가리켜 보라고 한다. 뼈이상은 아닌것 같으니 엑스레이를 찍을 필요는 없다는 건 합리적이라 좋았지만. 물리치료 받으라는데 믿음이 안가 그냥 나왔따'

In [5]:
suggestions_df['selected_match_score'].describe()


count    25.000000
mean      0.357580
std       0.167580
min       0.068408
25%       0.231702
50%       0.363672
75%       0.457171
max       0.687745
Name: selected_match_score, dtype: float64

### Notes
- Increase `max_length` if you want to keep more of long reviews for sentiment.
- For larger datasets, embed in batches and persist embeddings to disk.
- Suggestions are purely semantic similarity; consider adding type or alias constraints if needed.
